# Copilot Audit Log - Processor 
**Purpose.** Move the heavy Power Query transformations off the Power BI refresh path.
This notebook does, once in Spark, everything the `Chat + Agent Interactions (Audit Logs)`
Power Query used to do row-by-row in the single-threaded mashup engine:

* parse `AccessedResources` / `AISystemPlugin` JSON and flatten them,
* explode the accessed-resources list (1 interaction -> N resource rows),
* derive `InteractionDate` / `WeekStart` / `MonthStart`,
* normalise the UPN and left-join the licence flag,
* drop activity by non-human agent accounts (see `EXCLUDE_AGENT_IDENTITIES`),
* drop records that are not end-user agent usage (see `DROP_EXCLUDE_REASONS`),
* resolve `Agent_LinkID` by Title ID, then Bot Id, then Entra agent ID (never by agent name),
  giving the LOB and Shared copies of one Copilot Studio agent the same (LOB) Title ID.

It writes a single, flat, V-Ordered Delta table **`copilot_interactions_curated`** whose
column set is identical to the old Power Query output, so every calculated column,
measure and relationship in the model keeps working unchanged.

The table is stored by `MonthStart`. With `WRITE_MODE = "merge"` (the scheduled pipeline
default) a run recomputes only the months whose source rows changed, plus the months within
`INCREMENTAL_LOOKBACK_DAYS` of the newest record (late-arriving audit data), and replaces just
those months. Each recomputed month is rebuilt from all of its source rows, so it matches a
full rebuild. `WRITE_MODE = "overwrite"` rebuilds every month; a change to the settings, the
processing logic or the source schema does the same automatically, as does the first
`"merge"` run on a table that is not yet stored by month. Months that are not recomputed keep
the licence and agent-registry values from when they were last processed, and their
`Agent Last Used Date` is not refreshed; run `"overwrite"` to refresh everything.

Power BI then reads this table with **no transformation** (thin, foldable passthrough),
which is what makes both Direct Lake and a fast Incremental Refresh possible.

> Run this AFTER the audit-log ingester has produced `copilot_interactions_parsed`,
> and after the licensed-users and agents-365 producers have landed their tables.
> Schedule it in the same pipeline, immediately before the semantic-model refresh.

In [ ]:
# ----------------------------------------------------------------------------
# CONFIG
# ----------------------------------------------------------------------------
SRC_INTERACTIONS = "copilot_interactions_parsed"   # raw fact from the audit-log ingester
SRC_LICENSED     = "copilot_licensed_users"        # licensed-users dim (optional)
SRC_AGENTS       = "agents_365"                    # agents 365 dim (optional)

OUT_TABLE        = "copilot_interactions_curated"  # <-- Power BI reads this

# Full rebuild vs incremental. "overwrite" rebuilds the whole table (initial backfill, or
# to force a full rebuild). "merge" is the daily mode: the table is stored by MonthStart
# and only the months whose source rows changed, plus the months inside
# INCREMENTAL_LOOKBACK_DAYS of the newest record, are recomputed and replaced. The first
# "merge" run on a table that is not yet stored by month rebuilds it once (migration).
# Changing a setting below, the processing logic or the source schema also rebuilds.
# Requires stable key columns such as Id to be present.
WRITE_MODE       = "overwrite"                       # "overwrite" | "merge"
MERGE_KEYS       = ["Id"]                            # unique curated-row key column(s)
INCREMENTAL_LOOKBACK_DAYS = 14                       # always recompute months within this many days

RUN_OPTIMIZE     = True                              # OPTIMIZE + VORDER after write (touched months only)

# Opt in only after reviewing raw identity/resource data and downstream model exposure.
INCLUDE_RAW_PASSTHROUGH = False

# Drop activity by non-human agent accounts. Security Copilot agents, for example, sign in
# as SecurityCopilotAgentUser-<id>@<tenant>. They are not people, so they would inflate
# active users and sessions and top the "who to license next" list. Set False to keep them.
EXCLUDE_AGENT_IDENTITIES = True
AGENT_IDENTITY_PATTERNS  = [r"^securitycopilotagentuser-"]   # regexes on the lower-cased UPN

# Audit records that are not end-user agent usage, flagged in Exclude_Reason (see
# EXCLUDE_REASON_SQL). Rows with a reason listed here are removed; changing the list
# rebuilds the whole table on the next run. Delete a reason from the list to keep those rows; for
# example keep "Autonomous run" and "Workflow run" to report autonomous agent activity.
DROP_EXCLUDE_REASONS = [
    "Copilot Studio test pane",
    "Maker evaluation",
    "Agent authoring",
    "Autonomous run",
    "Workflow run",
    "M365 Copilot twin",
    "Fabric multi-agent",
]

# Optional agent-type overrides: a Lakehouse table with columns `key` and `Agent_Type`.
# key is an agent ID, agent name or AppIdentity (case-insensitive); Agent_Type replaces
# the rule-based category. Skipped when the table does not exist.
AGENT_TYPE_OVERRIDES_TABLE = "agent_type_overrides"

In [ ]:
from pyspark.sql import functions as F, types as T
from pyspark.sql import DataFrame
import re
import datetime
import hashlib
import json

# V-Order every Parquet/Delta file this session writes (required for Direct Lake perf).
spark.conf.set("spark.sql.parquet.vorder.enabled", "true")
# Let Spark evolve the Delta schema when tenant exports add/drop optional columns.
spark.conf.set("spark.databricks.delta.schema.autoMerge.enabled", "true")


def has_col(df: DataFrame, name: str) -> bool:
    return name in df.columns


def ensure_col(df: DataFrame, name: str, default=F.lit(None), dtype="string") -> DataFrame:
    # Idempotent 'add column if missing' -- mirrors the M Table.HasColumns guards.
    if has_col(df, name):
        return df
    return df.withColumn(name, default.cast(dtype))


def first_existing(df: DataFrame, candidates, fallback=None):
    for c in candidates:
        if c in df.columns:
            return c
    return fallback


RAW_PASSTHROUGH_COLUMNS = {
    "AppIdentity": "AppIdentity_Raw",
    "AccessedResources": "AccessedResources_Raw",
    "AISystemPlugin": "AISystemPlugin_Raw",
}


def retain_raw_payloads(df: DataFrame) -> DataFrame:
    # Reject aliases even under Spark/Delta's case-insensitive column resolution.
    existing = {c.casefold() for c in df.columns}
    collisions = [c for c in RAW_PASSTHROUGH_COLUMNS.values() if c.casefold() in existing]
    if collisions:
        raise ValueError(f"Raw passthrough output columns already exist: {collisions}")
    for source, target in RAW_PASSTHROUGH_COLUMNS.items():
        if source not in df.columns:
            df = df.withColumn(target, F.lit(None).cast("string"))
            continue
        value = F.col(source)
        if isinstance(df.schema[source].dataType, (T.StructType, T.ArrayType, T.MapType)):
            value = F.to_json(value, options={"ignoreNullFields": "false"})
        else:
            value = value.cast("string")
        df = df.withColumn(target, value)
    return df


def validate_passthrough_columns(source_columns, output_columns):
    # This is a presence guard, not a claim that canonical transformations are lossless.
    outputs = set(output_columns)
    missing = [c for c in source_columns
               if RAW_PASSTHROUGH_COLUMNS.get(c, c) not in outputs]
    if missing:
        raise RuntimeError(f"Raw passthrough source columns missing before write: {sorted(missing)}")


def normalize_actor_token(value) -> str:
    if value is None:
        return ""
    return " ".join(str(value).strip().lower().split())


def classify_actor_environment(app_host=None, environment=None, agent_name=None, agent_id=None, resource_type=None) -> str:
    host = normalize_actor_token(app_host)
    env = normalize_actor_token(environment)
    agent_name = normalize_actor_token(agent_name)
    agent_id = normalize_actor_token(agent_id)
    resource_type = normalize_actor_token(resource_type)
    if "cowork" in host or "cowork" in env or "cowork" in agent_name:
        return "Cowork"
    if host in ("autonomous", "logic app") or resource_type in ("flow", "connector") or agent_name or agent_id:
        return "Agents"
    return "User"


def resolve_write_strategy(write_mode, source_columns, target_exists, target_columns, merge_keys, source_key_health=True):
    mode = (write_mode or "").strip().lower()
    if mode not in ("overwrite", "merge"):
        raise ValueError(f"Unsupported WRITE_MODE={write_mode!r}")
    if mode == "overwrite":
        return "overwrite"
    source_columns = set(source_columns or [])
    target_columns = set(target_columns or [])
    missing_source = [k for k in merge_keys if k not in source_columns]
    if missing_source:
        raise RuntimeError(
            f"WRITE_MODE='merge' requires source key columns {missing_source}. "
            "Upgrade/rebuild the parsed audit source before enabling merge."
        )
    if not source_key_health:
        raise RuntimeError(
            f"WRITE_MODE='merge' requires non-blank values in merge keys {merge_keys}. "
            "Run overwrite after the ingester key upgrade if the legacy source cannot supply them."
        )
    if target_exists:
        missing_target = [k for k in merge_keys if k not in target_columns]
        if missing_target:
            raise RuntimeError(
                f"Existing table is missing merge key columns {missing_target}. "
                "Run WRITE_MODE='overwrite' once to initialise the curated table with the upgraded schema."
            )
        return "merge"
    return "overwrite"


def agent_identity_regex(enabled, patterns):
    # One regex for the agent accounts to drop, or None when the filter is off.
    parts = [p.strip() for p in (patterns or []) if p and p.strip()]
    if not enabled or not parts:
        return None
    return "|".join(f"(?:{p})" for p in parts)


# Records that are not end-user agent usage. Flagged with the first matching reason:
#   AppHost "Copilot Studio"       -> the maker's test pane
#   AppHost "pva-maker-evaluation" -> Copilot Studio evaluation runs
#   AppHost "agentic-builder"      -> agent authoring in Agent Builder
#   AppHost "autonomous"           -> autonomous (trigger-driven) agent runs
#   AppHost "workflow-agents"      -> workflow agent runs
#   no messages + AppHost "M365Copilot" -> the Copilot Studio runtime twin of an
#       M365 Copilot chat that is already counted from its own prompt record
#   all-zero AgentId               -> Fabric multi-agent orchestration
# Plain SQL (lower/trim/coalesce/replace) so the same text runs in Spark and in the tests.
EXCLUDE_REASON_SQL = """
CASE
  WHEN lower(trim(coalesce(AppHost, ''))) = 'copilot studio' THEN 'Copilot Studio test pane'
  WHEN lower(trim(coalesce(AppHost, ''))) = 'pva-maker-evaluation' THEN 'Maker evaluation'
  WHEN lower(trim(coalesce(AppHost, ''))) = 'agentic-builder' THEN 'Agent authoring'
  WHEN lower(trim(coalesce(AppHost, ''))) = 'autonomous' THEN 'Autonomous run'
  WHEN lower(trim(coalesce(AppHost, ''))) = 'workflow-agents' THEN 'Workflow run'
  WHEN upper(trim(coalesce(Prompts_Available, 'TRUE'))) = 'FALSE'
       AND lower(trim(coalesce(AppHost, ''))) = 'm365copilot' THEN 'M365 Copilot twin'
  WHEN trim(coalesce(AgentId, '')) <> ''
       AND replace(replace(lower(trim(coalesce(AgentId, ''))), '-', ''), '0', '') = ''
       THEN 'Fabric multi-agent'
  ELSE NULL
END
"""


# ---------------------------------------------------------------------------
# Agent linking. Pure Python: the registry is small enough to resolve on the driver.
# ---------------------------------------------------------------------------
PLATFORM_AGENT_ID_PATTERN = r'^(.*)_([0-9a-fA-F]{8}-[0-9a-fA-F]{4}-[0-9a-fA-F]{4}-[0-9a-fA-F]{4}-[0-9a-fA-F]{12})$'
BARE_GUID_PATTERN = r'^[0-9a-fA-F]{8}-[0-9a-fA-F]{4}-[0-9a-fA-F]{4}-[0-9a-fA-F]{4}-[0-9a-fA-F]{12}$'
# Declarative agent ids 'T_<title guid>.<schema guid>' tie a schema GUID to a Title ID.
TITLE_SCHEMA_PATTERN = r'^([TtPp]_[0-9a-fA-F]{8}-[0-9a-fA-F]{4}-[0-9a-fA-F]{4}-[0-9a-fA-F]{4}-[0-9a-fA-F]{12})\.([0-9a-fA-F]{8}-[0-9a-fA-F]{4}-[0-9a-fA-F]{4}-[0-9a-fA-F]{4}-[0-9a-fA-F]{12})$'


def _link_key(value) -> str:
    return "" if value is None else str(value).strip().lower()


def _letters(value) -> str:
    return re.sub(r"[^a-z]", "", _link_key(value))


def _is_zero_guid(value) -> bool:
    key = _link_key(value)
    return bool(key) and not key.replace("-", "").replace("0", "")


def _split_link_keys(value):
    parts = re.split(r"[;,]", "" if value is None else str(value))
    return [k for k in (_link_key(p) for p in parts) if k]


def preferred_agent(rows):
    # The LOB copy first, then Shared, then any other type; then the newest
    # Last updated; then the lowest Title ID, so the choice is deterministic.
    rank = {"lob": 0, "shared": 1}
    ordered = sorted(rows, key=lambda r: str(r.get("title") or "").strip())
    ordered = sorted(ordered, key=lambda r: str(r.get("updated") or ""), reverse=True)
    ordered = sorted(ordered, key=lambda r: rank.get(_link_key(r.get("type")), 2))
    return ordered[0]


def is_copilot_studio_copy(row) -> bool:
    # Only LOB/Shared Copilot Studio agents are merged. Agent Builder agents are
    # never merged: two of them with the same name are usually different agents.
    if _link_key(row.get("type")) not in ("lob", "shared"):
        return False
    created_in = _letters(row.get("created_in"))
    if "agentbuilder" in created_in:
        return False
    return bool(_split_link_keys(row.get("bot"))) or "copilotstudio" in created_in


def build_agent_link_maps(registry_rows):
    """Lookup maps for resolve_agent_link.

    registry_rows: dicts with title, bot, entra, type, created_in and updated.
    LOB and Shared copies of one Copilot Studio agent (same Bot Id or Entra agent
    ID) form a group whose canonical Title ID is the preferred (LOB) copy.
    """
    by_title = {}
    for row in registry_rows:
        title = str(row.get("title") or "").strip()
        if title:
            by_title.setdefault(title.upper(), []).append(dict(row, title=title))
    agents = {key: preferred_agent(rows) for key, rows in by_title.items()}

    bot_rows, entra_rows = {}, {}
    for agent in agents.values():
        for bot in _split_link_keys(agent.get("bot")):
            bot_rows.setdefault(bot, []).append(agent)
        entra = _link_key(agent.get("entra"))
        if entra and not _is_zero_guid(entra):
            entra_rows.setdefault(entra, []).append(agent)

    parent = {}

    def find(key):
        parent.setdefault(key, key)
        while parent[key] != key:
            parent[key] = parent[parent[key]]
            key = parent[key]
        return key

    for sharing in list(bot_rows.values()) + list(entra_rows.values()):
        members = [a["title"].upper() for a in sharing if is_copilot_studio_copy(a)]
        for other in members[1:]:
            parent[find(other)] = find(members[0])
    groups = {}
    for key in list(parent):
        groups.setdefault(find(key), []).append(agents[key])
    canonical = {}
    for members in groups.values():
        if len(members) > 1:
            lead = preferred_agent(members)["title"]
            for member in members:
                canonical[member["title"].upper()] = lead

    return {
        "title": {key: agent["title"] for key, agent in agents.items()},
        "bot": {key: preferred_agent(rows)["title"] for key, rows in bot_rows.items()},
        "entra": {key: preferred_agent(rows)["title"] for key, rows in entra_rows.items()},
        "canonical": canonical,
    }


def build_schema_title_map(agent_ids):
    # schema GUID -> Title ID, from 'T_<title guid>.<schema guid>' agent ids seen in
    # the audit log. Kept only when the schema GUID maps to exactly one Title ID.
    seen = {}
    for agent_id in agent_ids:
        match = re.match(TITLE_SCHEMA_PATTERN, "" if agent_id is None else str(agent_id).strip())
        if match:
            seen.setdefault(match.group(2).lower(), set()).add(match.group(1).upper())
    return {key: next(iter(titles)) for key, titles in seen.items() if len(titles) == 1}


def resolve_agent_link(maps, title_id=None, bot_id=None, entra_id=None, schema_title=None):
    """(Agent_LinkID, Agent_MatchedTitleID, Agent_LinkMethod) for one audit row.

    Keys are tried in order: Title ID, Bot Id, Entra agent ID, schema GUID. Agent
    names are never used. Agent_LinkMethod is 'Unlinked' when the row names an
    agent the registry does not hold, and None when the row has no agent key.
    """
    candidates = (
        ("Title ID", maps["title"].get(str(title_id or "").strip().upper())),
        ("Bot Id", maps["bot"].get(_link_key(bot_id))),
        ("Entra Agent ID", None if _is_zero_guid(entra_id) else maps["entra"].get(_link_key(entra_id))),
        ("Schema GUID", maps["title"].get(str(schema_title or "").strip().upper())),
    )
    for method, matched in candidates:
        if matched:
            return maps["canonical"].get(matched.upper(), matched), matched, method
    if any(_link_key(v) and not _is_zero_guid(v) for v in (title_id, bot_id, entra_id)):
        return None, None, "Unlinked"
    return None, None, None


# ---------------------------------------------------------------------------
# Agent type classification: what kind of agent a row used and who published it.
# Port of the rules in the Purview audit-log analyser. Pure Python: the processor
# runs it once per distinct agent key combination on the driver.
# Agent_Type_Basis says how firm each category is:
#   documented = a value pattern Microsoft documents for audit records
#   observed   = seen in real audit data but not formally documented
#   inferred   = the T_ / U_ / P_ agent-ID prefix convention, which Microsoft
#                does not document
#   override   = set by the optional AGENT_TYPE_OVERRIDES_TABLE
# ---------------------------------------------------------------------------
AGENT_TYPE_FP_PUBLISHED = "Microsoft first-party agents - published"
AGENT_TYPE_FP_NON_PUBLISHED = "Microsoft first-party agents - non-published"
AGENT_TYPE_FP_M365 = "Microsoft first-party agents - M365 Copilot agents"
AGENT_TYPE_DECLARATIVE = "Microsoft 365 Copilot Agent Builder - declarative agents"
AGENT_TYPE_CUSTOM_ENGINE = "Copilot Studio - custom engine agents"
AGENT_TYPE_COPILOT_STUDIO = "Copilot Studio - other / standalone agents"
AGENT_TYPE_FACILITATOR = "Microsoft Facilitator (Teams)"
AGENT_TYPE_ORG_PUBLISHED = "Published by your organisation"
AGENT_TYPE_SHARED = "Shared by creator"
AGENT_TYPE_STORE = "Agent Store package (publisher not identified)"
AGENT_TYPE_CONNECTED_APP = "Custom / third-party AI apps registered in your organisation"
AGENT_TYPE_UNCLASSIFIED = "Unclassified agents"

# Agent_Type -> (Agent_Publisher, Agent_Is_Published, Agent_Type_Basis).
# Agent_Is_Published is None where the audit log cannot tell.
AGENT_TYPE_INFO = {
    AGENT_TYPE_FP_PUBLISHED: ("Microsoft", True, "documented/observed"),
    AGENT_TYPE_FP_NON_PUBLISHED: ("Microsoft", False, "documented/observed"),
    AGENT_TYPE_FP_M365: ("Microsoft", True, "observed"),
    AGENT_TYPE_DECLARATIVE: ("Your organisation", None, "documented"),
    AGENT_TYPE_CUSTOM_ENGINE: ("Your organisation", None, "documented"),
    AGENT_TYPE_COPILOT_STUDIO: ("Your organisation", None, "documented"),
    AGENT_TYPE_FACILITATOR: ("Microsoft", True, "documented"),
    AGENT_TYPE_ORG_PUBLISHED: ("Your organisation", True, "inferred"),
    AGENT_TYPE_SHARED: ("User-shared", False, "inferred"),
    AGENT_TYPE_STORE: ("Agent Store", True, "inferred"),
    AGENT_TYPE_CONNECTED_APP: ("Connected app", None, "documented"),
    AGENT_TYPE_UNCLASSIFIED: ("Unknown", None, None),
}
FIRST_PARTY_AGENT_TYPES = (AGENT_TYPE_FP_PUBLISHED, AGENT_TYPE_FP_NON_PUBLISHED,
                           AGENT_TYPE_FP_M365, AGENT_TYPE_FACILITATOR)

# Microsoft agents that sometimes appear by name only. Extend as needed.
MICROSOFT_AGENT_NAMES = (
    "Researcher", "Analyst", "Facilitator", "Word Drafting Agent", "Draft as 1P Agent",
    "Prompt Coach", "Writing Coach", "Idea Coach", "Career Coach", "Learning Coach", "Surveys",
    "Skills Agent", "Employee Self-Service", "Visual Creator", "Project Manager",
    "Knowledge Agent", "Interpreter", "Sales Agent", "Service Agent", "People", "Photos",
)


def _agent_text(value) -> str:
    return "" if value is None else str(value).strip()


def _agent_name_key(value) -> str:
    # 'Word Drafting Agent', 'WordDraftingAgent' and 'word-drafting-agent' compare equal.
    return re.sub(r"[^a-z0-9]", "", _agent_text(value).lower())


MICROSOFT_AGENT_CANONICAL = {_agent_name_key(name): name for name in MICROSOFT_AGENT_NAMES}


def _microsoft_agent_segment(app_identity) -> str:
    # '<name>' of an AppIdentity 'MicrosoftAgent.<name>.<...>'.
    match = re.match(r"(?i)^microsoftagent\.([^.]+)", _agent_text(app_identity))
    return match.group(1) if match else ""


def microsoft_agent_name(agent_id=None, agent_name=None, app_identity=None):
    """Canonical Microsoft agent name ('Researcher', 'Word Drafting Agent', ...) or None.

    Tries the agent name, the agent ID (with and without a 'BuiltIn_' prefix) and the
    <name> part of 'MicrosoftAgent.<name>.<...>'.
    """
    aid = _agent_text(agent_id)
    candidates = [agent_name, aid, _microsoft_agent_segment(app_identity)]
    if aid.lower().startswith("builtin_"):
        candidates.append(aid[len("builtin_"):])
    for candidate in candidates:
        found = MICROSOFT_AGENT_CANONICAL.get(_agent_name_key(candidate))
        if found:
            return found
    return None


def build_agent_type_overrides(rows):
    """{lower-cased key: Agent_Type} from (key, Agent_Type) pairs. Blank pairs are skipped."""
    result = {}
    for key, agent_type in rows:
        key, agent_type = _agent_text(key).lower(), _agent_text(agent_type)
        if key and agent_type:
            result[key] = agent_type
    return result


def agent_type_override(overrides, agent_id=None, agent_name=None, app_identity=None):
    # Agent ID first, then agent name, then AppIdentity.
    for key in (agent_id, agent_name, app_identity):
        key = _agent_text(key).lower()
        if key and overrides and key in overrides:
            return overrides[key]
    return None


def classify_agent(agent_id=None, agent_name=None, app_identity=None, agent_type=None, workload=None):
    """Agent_Type for one row, or None when the row involves no agent. First match wins."""
    aid, name, app = _agent_text(agent_id), _agent_text(agent_name), _agent_text(app_identity)
    aid_l, app_l = aid.lower(), app.lower()
    kind = re.sub(r"\s", "", _agent_text(agent_type).lower())
    work = _agent_text(workload).lower()
    blob = f"{aid_l} {app_l}"
    if work == "aiapp" or app_l.startswith("aiapp."):
        return None  # third-party SaaS AI apps (for example ChatGPT) are not agents
    if work == "connectedaiapp" or app_l.startswith("connectedaiapp."):
        return AGENT_TYPE_CONNECTED_APP
    if app_l.startswith("copilot.teamcopilot."):
        return AGENT_TYPE_FACILITATOR
    if "customengine" in blob:
        return AGENT_TYPE_CUSTOM_ENGINE
    if "copilotstudio.declarative" in blob or "copilot.studio.declarative" in blob:
        return AGENT_TYPE_DECLARATIVE
    if app_l.startswith("microsoftagent.") or aid_l.startswith("builtin_") or microsoft_agent_name(aid, name, app):
        if app_l.startswith("copilot.m365copilot"):
            return AGENT_TYPE_FP_M365
        if app_l.startswith("microsoftagent."):
            published = re.match(r"^microsoftagent\..+\.p_", app_l) is not None
        else:
            published = aid_l.startswith("p_")
        return AGENT_TYPE_FP_PUBLISHED if published else AGENT_TYPE_FP_NON_PUBLISHED
    if app_l.startswith("copilot.studio.") or kind == "copilotstudio":
        return AGENT_TYPE_COPILOT_STUDIO
    if not (aid or name or kind):
        return None  # plain Copilot chat: no agent fields
    prefix = (aid or app)[:2].upper()
    if prefix == "T_":
        return AGENT_TYPE_ORG_PUBLISHED
    if prefix == "U_":
        return AGENT_TYPE_SHARED
    if prefix == "P_":
        return AGENT_TYPE_STORE
    # An agent inside the Microsoft 365 Copilot app that no rule above identified.
    if app_l.startswith("copilot.m365copilot"):
        return AGENT_TYPE_FP_M365
    return AGENT_TYPE_UNCLASSIFIED


def describe_agent(agent_id=None, agent_name=None, app_identity=None, agent_type=None, workload=None,
                   overrides=None):
    """(Agent_Key, Agent_Type, Agent_Type_Basis, Agent_Publisher, Agent_Is_Published,
    Agent_Consolidated_Name) for one row. All None when the row involves no agent.

    Agent_Consolidated_Name rolls every Microsoft first-party row up to one canonical
    name (for example all Researcher rows, whatever their ID or host); other agents
    keep their own name.
    """
    aid, name, app = _agent_text(agent_id), _agent_text(agent_name), _agent_text(app_identity)
    override = agent_type_override(overrides, aid, name, app)
    category = override or classify_agent(aid, name, app, agent_type, workload)
    if not category:
        return (None,) * 6
    publisher, published, basis = AGENT_TYPE_INFO.get(category, ("Unknown", None, None))
    if override:
        basis = "override"
    consolidated = None
    if category in FIRST_PARTY_AGENT_TYPES:
        consolidated = (microsoft_agent_name(aid, name, app)
                        or ("Facilitator" if category == AGENT_TYPE_FACILITATOR else None)
                        or name or _microsoft_agent_segment(app))
    return (aid or name or app or None, category, basis, publisher, published,
            consolidated or name or aid or app or None)

# --- Incremental processing (verbatim from core/valuelens_core/incremental.py) -----
# The curated table is stored by MonthStart. A run recomputes only the months whose
# source rows changed plus the lookback months; see plan_incremental_months.
# Bump PROCESSOR_LOGIC_VERSION whenever a change to cells 2-14 alters curated rows,
# so the next run rebuilds every month (tests/test_incremental_processing.py checks it).
PROCESSOR_LOGIC_VERSION = 1
PARTITION_COLUMN = "MonthStart"
PROCESSOR_STATE_PROPERTY = "valuelens.processor.state"

PROCESSOR_STATE_VERSION = 1


NULL_MONTH = "none"


DEFAULT_INCREMENTAL_LOOKBACK_DAYS = 14


def month_key(value) -> str:
    # 'YYYY-MM' for a date, datetime or ISO text; NULL_MONTH for rows without a usable date.
    if value is None:
        return NULL_MONTH
    if isinstance(value, (datetime.date, datetime.datetime)):
        return f"{value.year:04d}-{value.month:02d}"
    text = str(value).strip()
    if len(text) >= 7 and text[4] == "-" and text[:4].isdigit() and text[5:7].isdigit():
        return text[:7]
    return NULL_MONTH


def month_start(key):
    # First day of a month key; None for NULL_MONTH.
    if key == NULL_MONTH:
        return None
    return datetime.date(int(key[:4]), int(key[5:7]), 1)


def processor_config_fingerprint(settings) -> str:
    # Stable hash of everything that changes the output of every month.
    payload = json.dumps(settings, sort_keys=True, default=str, separators=(",", ":"))
    return hashlib.sha256(payload.encode("utf-8")).hexdigest()


def parse_processor_state(text):
    # The state saved by the last successful run, or None when absent or unreadable.
    if not text:
        return None
    try:
        state = json.loads(text)
    except (TypeError, ValueError):
        return None
    if (not isinstance(state, dict) or state.get("version") != PROCESSOR_STATE_VERSION
            or not isinstance(state.get("config"), str) or not isinstance(state.get("months"), dict)):
        return None
    return state


def build_processor_state(config, months) -> str:
    return json.dumps({"version": PROCESSOR_STATE_VERSION, "config": config,
                       "months": {str(k): str(v) for k, v in sorted(months.items())}},
                      sort_keys=True, separators=(",", ":"))


def month_predicate_sql(column, months) -> str:
    # SQL predicate selecting the given month keys of a MonthStart-style DATE column.
    dates = sorted(month_start(k).isoformat() for k in months if k != NULL_MONTH)
    parts = []
    if dates:
        parts.append(f"`{column}` IN ({', '.join(f'DATE{chr(39)}{d}{chr(39)}' for d in dates)})")
    if NULL_MONTH in months:
        parts.append(f"`{column}` IS NULL")
    return "(" + " OR ".join(parts) + ")" if parts else "FALSE"


def plan_incremental_months(source_months, state, config, *, max_creation=None,
                            lookback_days=DEFAULT_INCREMENTAL_LOOKBACK_DAYS,
                            force_full=False, target_ready=True):
    """Which months to (re)compute.

    source_months maps month key -> fingerprint of the source rows in that month.
    Returns {"full": bool, "reason": str, "months": sorted month keys}. On a full
    rebuild "months" lists every source month. Otherwise it lists the changed
    months, the months inside the lookback window and the months that vanished
    from the source (rewritten empty, as a full rebuild would drop them).
    """
    source_months = {str(k): str(v) for k, v in (source_months or {}).items()}
    everything = sorted(source_months)
    if force_full:
        return {"full": True, "reason": "full rebuild requested", "months": everything}
    if not target_ready:
        return {"full": True, "reason": "target missing or not stored by month (one-time migration)",
                "months": everything}
    if state is None:
        return {"full": True, "reason": "no saved processing state", "months": everything}
    if state.get("config") != config:
        return {"full": True, "reason": "processor settings, logic or source schema changed",
                "months": everything}
    saved = {str(k): str(v) for k, v in state["months"].items()}
    months = {k for k, v in source_months.items() if saved.get(k) != v}
    months |= {k for k in saved if k not in source_months}
    if max_creation is not None:
        if isinstance(max_creation, datetime.datetime):
            max_creation = max_creation.date()
        floor = month_key(max_creation - datetime.timedelta(days=max(int(lookback_days or 0), 0)))
        months |= {k for k in source_months if k != NULL_MONTH and k >= floor}
    reason = "incremental: no changed months" if not months else "incremental"
    return {"full": False, "reason": reason, "months": sorted(months)}


In [ ]:
# ----------------------------------------------------------------------------
# 1. READ RAW FACT + PLAN WHICH MONTHS TO PROCESS
#    One pass over the source fingerprints every month (row count + row hash).
#    WRITE_MODE="merge" recomputes only changed months and the lookback months;
#    "overwrite" (or a settings/logic/schema change) rebuilds every month.
# ----------------------------------------------------------------------------
src_all = spark.table(SRC_INTERACTIONS)
passthrough_source_columns = list(src_all.columns)

_src_month_col = (F.date_format(F.trunc(F.to_date(F.to_timestamp(F.col("CreationDate"))), "month"), "yyyy-MM")
                  if has_col(src_all, "CreationDate") else F.lit(None).cast("string"))
src_all = src_all.withColumn("__vl_month", F.coalesce(_src_month_col, F.lit(NULL_MONTH)))
_hash_cols = [F.col(f"`{f.name}`") for f in src_all.schema.fields
              if f.name != "__vl_month" and "map<" not in f.dataType.simpleString()]  # hashing maps is unsupported
_blank_key = F.lit(False)
for _k in MERGE_KEYS:
    if has_col(src_all, _k):
        _blank_key = _blank_key | F.col(_k).isNull() | (F.trim(F.col(_k).cast("string")) == "")
_month_stats = (src_all.groupBy("__vl_month").agg(
        F.count(F.lit(1)).alias("rows"),
        F.sum(F.xxhash64(*_hash_cols).cast("decimal(38,0)")).alias("hash"),
        F.max(F.to_timestamp(F.col("CreationDate")) if has_col(src_all, "CreationDate")
              else F.lit(None).cast("timestamp")).alias("max_ts"),
        F.sum(F.when(_blank_key, 1).otherwise(0)).alias("blank_keys"))
    .collect())
SOURCE_MONTHS = {r["__vl_month"]: f"{r['rows']}:{r['hash']}" for r in _month_stats}
SOURCE_ROWS = sum(r["rows"] for r in _month_stats)
_max_ts = [r["max_ts"] for r in _month_stats if r["max_ts"] is not None]
SOURCE_MAX_CREATION = max(_max_ts) if _max_ts else None

# Everything that changes every month's output. Dimension tables (licences, agents) are
# not included: months that are not recomputed keep the values from when they were.
_overrides_rows = []
if AGENT_TYPE_OVERRIDES_TABLE and spark.catalog.tableExists(AGENT_TYPE_OVERRIDES_TABLE):
    _overrides_rows = sorted([[str(v) for v in r] for r in spark.table(AGENT_TYPE_OVERRIDES_TABLE).collect()])
PROCESSOR_CONFIG = processor_config_fingerprint({
    "logic": PROCESSOR_LOGIC_VERSION,
    "source": [[f.name, f.dataType.simpleString()] for f in spark.table(SRC_INTERACTIONS).schema.fields],
    "include_raw_passthrough": bool(INCLUDE_RAW_PASSTHROUGH),
    "exclude_agent_identities": bool(EXCLUDE_AGENT_IDENTITIES),
    "agent_identity_patterns": list(AGENT_IDENTITY_PATTERNS or []),
    "drop_exclude_reasons": sorted(DROP_EXCLUDE_REASONS or []),
    "agent_type_overrides": _overrides_rows,
})

_target_ready, _state = False, None
if WRITE_MODE.strip().lower() == "merge":
    _target_exists = spark.catalog.tableExists(OUT_TABLE)
    _target_columns = spark.table(OUT_TABLE).columns if _target_exists else []
    resolve_write_strategy(WRITE_MODE, src_all.columns, _target_exists, _target_columns, MERGE_KEYS,
                           source_key_health=sum(r["blank_keys"] for r in _month_stats) == 0)
    if _target_exists:
        _detail = spark.sql(f"DESCRIBE DETAIL {OUT_TABLE}").collect()[0]
        _target_ready = list(_detail["partitionColumns"] or []) == [PARTITION_COLUMN]
        _state = parse_processor_state((_detail["properties"] or {}).get(PROCESSOR_STATE_PROPERTY))
else:
    resolve_write_strategy(WRITE_MODE, src_all.columns, False, [], MERGE_KEYS)

PROCESS_PLAN = plan_incremental_months(
    SOURCE_MONTHS, _state, PROCESSOR_CONFIG, max_creation=SOURCE_MAX_CREATION,
    lookback_days=INCREMENTAL_LOOKBACK_DAYS,
    force_full=WRITE_MODE.strip().lower() != "merge", target_ready=_target_ready)
print(f"plan: {'full rebuild' if PROCESS_PLAN['full'] else 'incremental'} ({PROCESS_PLAN['reason']}); "
      f"{len(PROCESS_PLAN['months'])} of {len(SOURCE_MONTHS)} month(s): {', '.join(PROCESS_PLAN['months'][:24])}"
      + (" ..." if len(PROCESS_PLAN['months']) > 24 else ""))

# Context from the months that are kept, so cross-month values match a full rebuild:
# schema-GUID agent IDs (agent link) and the latest use of each agent name.
KEPT_SCHEMA_AGENT_IDS, KEPT_AGENT_LAST_USED = [], []
if PROCESS_PLAN["full"]:
    fact = src_all
else:
    _months = set(PROCESS_PLAN["months"])
    fact = src_all.where(F.col("__vl_month").isin(*sorted(_months)) if _months else F.lit(False))
    _kept = spark.table(OUT_TABLE).where(~F.expr(month_predicate_sql(PARTITION_COLUMN, _months)))
    if has_col(_kept, "AgentId"):
        KEPT_SCHEMA_AGENT_IDS = [r[0] for r in _kept.where(F.col("AgentId").rlike(TITLE_SCHEMA_PATTERN))
                                 .select("AgentId").distinct().collect()]
    if has_col(_kept, "AgentName"):
        KEPT_AGENT_LAST_USED = [(r[0], r[1]) for r in _kept
            .where(F.col("AgentName").isNotNull() & (F.trim(F.col("AgentName").cast("string")) != ""))
            .groupBy("AgentName").agg(F.max("CreationDate")).collect()]
fact = fact.drop("__vl_month")

if INCLUDE_RAW_PASSTHROUGH:
    fact = retain_raw_payloads(fact)
print(f"{SRC_INTERACTIONS}: {SOURCE_ROWS:,} rows, {len(passthrough_source_columns)} cols")


In [ ]:
# ----------------------------------------------------------------------------
# 2. ENSURE OPTIONAL COLUMNS EXIST (tenant exports vary)
#    Mirrors the 'Add ...' idempotent steps in the Power Query.
# ----------------------------------------------------------------------------
fact = ensure_col(fact, "Message_isPrompt", F.lit("TRUE"))
fact = ensure_col(fact, "ModelTransparencyDetails_ModelProviderName")
fact = ensure_col(fact, "ModelTransparencyDetails_ModelName")
fact = ensure_col(fact, "ApplicationName")
fact = ensure_col(fact, "Audit_UserKey")
fact = ensure_col(fact, "SensitivityLabelId")
fact = ensure_col(fact, "AccessedResource_SensitivityLabelId")
for _key_col in ["Id", "RecordId", "Source_RecordKey", "Source_MessageKey", "Source_ResourceKey"]:
    fact = ensure_col(fact, _key_col)

# Agent-type rule inputs. Parsed tables written before the ingester kept them lack them.
for _c in ["AppIdentity_Text", "Agent_TargetPlatformId", "Agent_TargetName", "Agent_PlatformType"]:
    fact = ensure_col(fact, _c)
# AppIdentity is documented as plain text ('Copilot.Studio.<AppId>'); keep that for the rules.
if has_col(fact, "AppIdentity") and isinstance(fact.schema["AppIdentity"].dataType, T.StringType):
    fact = fact.withColumn("AppIdentity_Text", F.coalesce(F.col("AppIdentity_Text"), F.col("AppIdentity")))

# Keep canonical conversion unchanged; opt-in raw copy was captured before shaping.
fact = ensure_col(fact, "AppIdentity_AppId")
if has_col(fact, "AppIdentity"):
    if not has_col(fact, "AppIdentity_DisplayName"):
        fact = fact.withColumn("AppIdentity_DisplayName", F.col("AppIdentity").cast("string"))
    fact = fact.drop("AppIdentity")
else:
    fact = ensure_col(fact, "AppIdentity_DisplayName")

# Copilot Studio runtime keys and the exclusion flag. Older parsed tables lack them.
for _c in ["AgentId", "AppHost", "Agent_EntraId", "AgentPlatform", "PlatformAgentId",
           "Agent_BotId", "Agent_EnvironmentId"]:
    fact = ensure_col(fact, _c)
fact = ensure_col(fact, "Prompts_Available", F.lit("TRUE"))
fact = fact.withColumn("Prompts_Available",
                       F.coalesce(F.upper(F.trim(F.col("Prompts_Available").cast("string"))), F.lit("TRUE")))


def _nonblank_text(c):
    t = F.trim(F.col(c).cast("string"))
    return F.when(t.isNotNull() & (t != ""), t)


_pid = F.trim(F.coalesce(F.col("PlatformAgentId").cast("string"), F.lit("")))
_aid = F.trim(F.coalesce(F.col("AgentId").cast("string"), F.lit("")))
fact = (fact
    .withColumn("Agent_BotId", F.lower(F.coalesce(
        _nonblank_text("Agent_BotId"),
        F.when(_pid.rlike(PLATFORM_AGENT_ID_PATTERN), F.regexp_extract(_pid, PLATFORM_AGENT_ID_PATTERN, 2))
         .when(_pid.rlike(BARE_GUID_PATTERN), _pid))))
    .withColumn("Agent_EnvironmentId", F.coalesce(
        _nonblank_text("Agent_EnvironmentId"),
        F.when(_pid.rlike(PLATFORM_AGENT_ID_PATTERN), F.regexp_extract(_pid, PLATFORM_AGENT_ID_PATTERN, 1))))
    .withColumn("Agent_EntraId", F.coalesce(
        _nonblank_text("Agent_EntraId"),
        F.when(_aid.rlike(BARE_GUID_PATTERN), _aid))))
# An all-zero GUID (Fabric multi-agent orchestration) is not an agent identity.
fact = fact.withColumn("Agent_EntraId",
                       F.when(F.regexp_replace(F.coalesce(F.col("Agent_EntraId"), F.lit("")), "[-0]", "") == "",
                              F.lit(None).cast("string")).otherwise(F.col("Agent_EntraId")))

# Recomputed here so older parsed rows are classified the same way as new ones.
fact = fact.withColumn("Exclude_Reason", F.expr(EXCLUDE_REASON_SQL))
if DROP_EXCLUDE_REASONS:
    _excluded = F.coalesce(F.col("Exclude_Reason").isin(*DROP_EXCLUDE_REASONS), F.lit(False))
    for _r in fact.where(_excluded).groupBy("Exclude_Reason").count().orderBy("Exclude_Reason").collect():
        print(f"excluded: {_r['count']:,} rows  ({_r['Exclude_Reason']})")
    fact = fact.where(~_excluded)

In [ ]:
# ----------------------------------------------------------------------------
# 3. ACCESSED RESOURCES  --  parse JSON + explode (this was the big fold-breaker)
#    Old M: Json.Document -> Table.ExpandListColumn -> Table.ExpandRecordColumn.
#    Here: from_json to an array<struct>, explode_outer, then project.
# ----------------------------------------------------------------------------
ar_schema = T.ArrayType(T.StructType([
    T.StructField("Type",    T.StringType()),
    T.StructField("Action",  T.StringType()),
    T.StructField("SiteUrl", T.StringType()),
]))

if has_col(fact, "AccessedResources"):
    parsed = fact.withColumn(
        "_resources",
        F.when(
            (F.col("AccessedResources").isNull()) | (F.trim(F.col("AccessedResources")) == ""),
            F.array().cast(ar_schema),
        ).otherwise(F.coalesce(F.from_json(F.col("AccessedResources"), ar_schema),
                               F.array().cast(ar_schema))),
    ).drop("AccessedResources")

    # explode_outer keeps interactions that have zero accessed resources (LEFT semantics).
    parsed = parsed.withColumn("_res", F.explode_outer("_resources")).drop("_resources")
    fact = (parsed
            .withColumn("AccessedResource_Type",    F.col("_res.Type"))
            .withColumn("AccessedResource_Action",  F.col("_res.Action"))
            .withColumn("AccessedResource_SiteUrl", F.col("_res.SiteUrl"))
            .drop("_res"))
else:
    fact = ensure_col(fact, "AccessedResource_Type")
    fact = ensure_col(fact, "AccessedResource_Action")
    fact = ensure_col(fact, "AccessedResource_SiteUrl")

In [ ]:
# ----------------------------------------------------------------------------
# 4. AI SYSTEM PLUGIN  --  parse JSON, take first element if it is a list
# ----------------------------------------------------------------------------
plugin_obj = T.StructType([
    T.StructField("Id",   T.StringType()),
    T.StructField("Name", T.StringType()),
])
plugin_arr = T.ArrayType(plugin_obj)

if has_col(fact, "AISystemPlugin"):
    raw = F.col("AISystemPlugin")
    as_arr = F.from_json(raw, plugin_arr)
    as_obj = F.from_json(raw, plugin_obj)
    single = F.when(as_arr.isNotNull() & (F.size(as_arr) > 0), as_arr.getItem(0)).otherwise(as_obj)
    fact = (fact
            .withColumn("_plugin", F.when((raw.isNull()) | (F.trim(raw) == ""), F.lit(None).cast(plugin_obj)).otherwise(single))
            .withColumn("AISystemPlugin_Id",   F.col("_plugin.Id"))
            .withColumn("AISystemPlugin_Name", F.col("_plugin.Name"))
            .drop("_plugin").drop("AISystemPlugin"))
else:
    fact = ensure_col(fact, "AISystemPlugin_Id")
    fact = ensure_col(fact, "AISystemPlugin_Name")

In [ ]:
# ----------------------------------------------------------------------------
# 5. DATES + RESOURCE COUNT
# ----------------------------------------------------------------------------
fact = fact.withColumn("CreationDate", F.to_timestamp("CreationDate"))

for c in ["InteractionDate", "WeekStart", "MonthStart"]:
    if has_col(fact, c):
        fact = fact.drop(c)

fact = (fact
        .withColumn("InteractionDate", F.to_date("CreationDate"))
        # Power Query used Day.Monday as the week start.
        .withColumn("WeekStart", F.date_sub(F.next_day(F.col("InteractionDate"), "Mon"), 7))
        .withColumn("MonthStart", F.trunc(F.col("InteractionDate"), "month")))

if has_col(fact, "Resource_Count"):
    fact = fact.withColumn("Resource_Count", F.col("Resource_Count").cast("long"))
else:
    fact = fact.withColumn("Resource_Count", F.lit(1).cast("long"))

In [ ]:
# ----------------------------------------------------------------------------
# 6. NORMALISED UPN  (join key to the licensed-users dim)
# ----------------------------------------------------------------------------
if has_col(fact, "Audit_UserId_Normalized"):
    norm = F.coalesce(F.col("Audit_UserId_Normalized"),
                      F.lower(F.trim(F.col("Audit_UserId").cast("string"))))
elif has_col(fact, "Audit_UserId"):
    norm = F.lower(F.trim(F.col("Audit_UserId").cast("string")))
else:
    norm = F.lit(None).cast("string")
fact = fact.withColumn("_NormUPN", norm)

# ----------------------------------------------------------------------------
# 6b. AGENT IDENTITIES  (not people; see EXCLUDE_AGENT_IDENTITIES)
# ----------------------------------------------------------------------------
AGENT_IDENTITY_REGEX = agent_identity_regex(EXCLUDE_AGENT_IDENTITIES, AGENT_IDENTITY_PATTERNS)
if AGENT_IDENTITY_REGEX:
    _is_agent = F.coalesce(F.lower(F.col("_NormUPN")).rlike(AGENT_IDENTITY_REGEX), F.lit(False))
    _dropped = fact.where(_is_agent).count()
    fact = fact.where(~_is_agent)
    print(f"agent identities: dropped {_dropped:,} rows (set EXCLUDE_AGENT_IDENTITIES = False to keep them)")


In [ ]:
# ----------------------------------------------------------------------------
# 7. LEFT-JOIN LICENCE FLAG  (dedup dim first -> no row fan-out)
# ----------------------------------------------------------------------------
def load_licensed():
    try:
        lic = spark.table(SRC_LICENSED)
    except Exception as e:
        print(f"[warn] {SRC_LICENSED} not found ({e}); 'Has license' will be null.")
        return None
    upn_col = first_existing(lic, ["User Principal Name", "userPrincipalName",
                                   "UserPrincipalName", "User principal name",
                                   "User_Principal_Name"])
    has_lic = first_existing(lic, ["Has license", "Has License", "HasLicense", "HasCopilot",
                                   "Has Copilot", "Has Copilot License", "HasCopilotLicense",
                                   "isUser", "Has_license"])
    if has_col(lic, "UPN_Normalized"):
        key = F.col("UPN_Normalized")
    elif upn_col:
        key = F.lower(F.trim(F.col(upn_col).cast("string")))
    else:
        return None
    hl = F.col(has_lic).cast("string") if has_lic else F.lit("Unknown")
    lic = (lic.withColumn("UPN_Normalized", key)
              .withColumn("Has license", hl)
              .where(F.col("UPN_Normalized").isNotNull() & (F.trim("UPN_Normalized") != ""))
              .select("UPN_Normalized", "Has license")
              .dropDuplicates(["UPN_Normalized"]))
    return lic

lic = load_licensed()
if lic is not None:
    fact = fact.join(F.broadcast(lic), fact["_NormUPN"] == lic["UPN_Normalized"], "left") \
               .drop(lic["UPN_Normalized"])
else:
    fact = ensure_col(fact, "Has license")

In [ ]:
# ----------------------------------------------------------------------------
# 8. AGENT_TITLEID  (keep existing; else derive text before first '.' of AgentId)
# ----------------------------------------------------------------------------
aid = F.trim(F.col("AgentId").cast("string")) if has_col(fact, "AgentId") else F.lit(None).cast("string")
derived = F.when(aid.isNull() | (aid == ""), F.lit(None)) \
           .otherwise(F.when(F.instr(aid, ".") > 0, F.substring_index(aid, ".", 1)).otherwise(aid))
if has_col(fact, "Agent_TitleID"):
    existing = F.trim(F.col("Agent_TitleID").cast("string"))
    fact = fact.withColumn("Agent_TitleID",
                           F.when(existing.isNotNull() & (existing != ""), existing).otherwise(derived))
else:
    fact = fact.withColumn("Agent_TitleID", derived)

fact = ensure_col(fact, "Agent_EntraId")
fact = ensure_col(fact, "AgentName")

In [ ]:
# ----------------------------------------------------------------------------
# 9. AGENT LINK  (Title ID -> Bot Id -> Entra agent ID -> schema GUID; never by name)
#    Agent_LinkID is the registry Title ID a row belongs to. The LOB and Shared copies
#    of one Copilot Studio agent (same Bot Id or Entra agent ID) share the LOB copy's
#    Title ID, so distinct users are counted once across the copies. Agent Builder
#    copies are never merged. Agent_MatchedTitleID keeps the Title ID that matched
#    before merging; Agent_LinkMethod says which key matched.
# ----------------------------------------------------------------------------
def load_agents():
    try:
        return spark.table(SRC_AGENTS)
    except Exception as e:
        print(f"[warn] {SRC_AGENTS} not found ({e}); Agent_LinkID falls back to null.")
        return None

ag = load_agents()
fact = (fact
    .withColumn("__lk_title", F.upper(_nonblank_text("Agent_TitleID")))
    .withColumn("__lk_bot", F.lower(_nonblank_text("Agent_BotId")))
    .withColumn("__lk_entra", F.lower(_nonblank_text("Agent_EntraId"))))
fact = fact.drop("Agent_LinkID", "Agent_MatchedTitleID", "Agent_LinkMethod")

_registry_rows = []
if ag is not None and "Title ID" in ag.columns:
    def _reg(names):
        c = first_existing(ag, names)
        return F.col(c).cast("string") if c else F.lit(None).cast("string")
    _registry_rows = [r.asDict() for r in ag.select(
        _reg(["Title ID"]).alias("title"),
        _reg(["Bot Id", "BotId", "Bot ID"]).alias("bot"),
        _reg(["Entra Agent ID", "Entra agent ID", "EntraAgentId", "Entra Agent Id",
              "EntraAgentID", "Agent ID", "AgentId", "Agent Id"]).alias("entra"),
        _reg(["Agent type (A365)", "Agent Type (A365)", "Agent type"]).alias("type"),
        _reg(["Created in", "Platform"]).alias("created_in"),
        _reg(["Last updated", "Last Updated"]).alias("updated"),
    ).collect()]

_maps = build_agent_link_maps(_registry_rows)
# KEPT_SCHEMA_AGENT_IDS: schema-GUID agent IDs from months this run does not recompute.
_schema_titles = build_schema_title_map(
    [r[0] for r in fact.where(F.col("AgentId").rlike(TITLE_SCHEMA_PATTERN)).select("AgentId").distinct().collect()]
    + list(globals().get("KEPT_SCHEMA_AGENT_IDS") or [])
) if has_col(fact, "AgentId") else {}

# Resolve each distinct key combination once on the driver, then broadcast-join back.
_links = []
for r in fact.select("__lk_title", "__lk_bot", "__lk_entra").distinct().collect():
    _schema = _schema_titles.get(r[2]) if r[2] and re.match(BARE_GUID_PATTERN, r[2]) else None
    _links.append((r[0], r[1], r[2]) + tuple(resolve_agent_link(_maps, r[0], r[1], r[2], _schema)))
_link_df = spark.createDataFrame(
    _links,
    "_lk_title string, _lk_bot string, _lk_entra string, "
    "Agent_LinkID string, Agent_MatchedTitleID string, Agent_LinkMethod string")
_on = (fact["__lk_title"].eqNullSafe(_link_df["_lk_title"])
       & fact["__lk_bot"].eqNullSafe(_link_df["_lk_bot"])
       & fact["__lk_entra"].eqNullSafe(_link_df["_lk_entra"]))
fact = fact.join(F.broadcast(_link_df), _on, "left").drop("_lk_title", "_lk_bot", "_lk_entra")

_methods = {}
for _l in _links:
    _methods[_l[5]] = _methods.get(_l[5], 0) + 1
print(f"agent link: {len(_maps['title']):,} registry agents, "
      f"{len(set(_maps['canonical'].values())):,} LOB/Shared groups; key combinations by method: "
      + ", ".join(f"{k}={v:,}" for k, v in sorted(_methods.items(), key=lambda kv: str(kv[0]))))

fact = fact.drop("__lk_title", "__lk_bot", "__lk_entra", "_NormUPN")
if not INCLUDE_RAW_PASSTHROUGH:
    fact = fact.drop("Audit_UserId_Normalized")

In [ ]:
# ----------------------------------------------------------------------------
# 10. GUARANTEE THE MODEL CONTRACT
#     The semantic model binds to these 33 sourced columns. A tenant export can
#     omit optional ones, so add any missing as typed nulls -- the import then
#     never errors 'column not found', exactly like the old Power Query guards.
# ----------------------------------------------------------------------------
REQUIRED_TEXT_COLS = [
    "AISystemPlugin_Id", "AISystemPlugin_Name",
    "AccessedResource_Action", "AccessedResource_SensitivityLabelId",
    "AccessedResource_SiteUrl", "AccessedResource_Type",
    "AgentId", "AgentName", "Agent_EntraId", "Agent_LinkID", "Agent_TitleID",
    "AppHost", "AppIdentity_AppId", "AppIdentity_DisplayName", "AppIdentity_PublisherId",
    "ApplicationName", "Audit_UserId", "Audit_UserKey", "ClientRegion", "Context_Type",
    "Has license", "Message_Id", "Message_isPrompt",
    "ModelTransparencyDetails_ModelName", "ModelTransparencyDetails_ModelProviderName",
    "SensitivityLabelId", "ThreadId", "Workload",
]
# Agent-link and runtime columns added after the original contract.
AGENT_LINK_TEXT_COLS = [
    "Agent_BotId", "Agent_EnvironmentId", "Agent_MatchedTitleID", "Agent_LinkMethod",
    "AgentPlatform", "PlatformAgentId", "Prompts_Available", "Exclude_Reason",
]
for c in REQUIRED_TEXT_COLS + AGENT_LINK_TEXT_COLS:
    fact = ensure_col(fact, c)                    # text, null when absent

# Non-text contract columns are always produced above, but keep types explicit.
fact = fact.withColumn("CreationDate", F.col("CreationDate").cast("timestamp")) \
           .withColumn("InteractionDate", F.col("InteractionDate").cast("date")) \
           .withColumn("WeekStart", F.col("WeekStart").cast("date")) \
           .withColumn("MonthStart", F.col("MonthStart").cast("date")) \
           .withColumn("Resource_Count", F.col("Resource_Count").cast("long"))

missing = [c for c in (REQUIRED_TEXT_COLS + AGENT_LINK_TEXT_COLS + ["CreationDate","InteractionDate","WeekStart","MonthStart","Resource_Count"]) if c not in fact.columns]
assert not missing, f"model contract broken, missing: {missing}"
print(f"model contract satisfied: all 33 sourced columns present (+{len(AGENT_LINK_TEXT_COLS)} agent-link columns)")

## Behaviour enrichment (was 26 DAX calculated columns)
Ported from the template's calculated columns so the curated table carries them and the
semantic model imports with **zero calculated columns** -> fast Import+Incremental Refresh
(Pro-safe) and Direct Lake-eligible. Runs after the contract guard, before the Delta write.

In [ ]:
# ----------------------------------------------------------------------------
# 10b. BEHAVIOUR ENRICHMENT  (ported from the .pbit DAX calculated columns)
#      Produces the 32 columns the semantic model used to compute at refresh
#      time, so the fact table imports with ZERO calculated columns:
#        - Import + Incremental Refresh stays fast (no per-row DAX)
#        - model becomes Direct Lake-eligible (calc columns block Direct Lake)
#      Row-level and deterministic -> identical output to the old DAX.
# ----------------------------------------------------------------------------
from pyspark.sql import Column, Window

def L(v):
    return v if isinstance(v, Column) else F.lit(v)

def chain(pairs, default):
    e = None
    for cond, val in pairs:
        e = F.when(cond, L(val)) if e is None else e.when(cond, L(val))
    return e.otherwise(L(default))

def _s(c):        # lower+trim, null -> ""   (for case-insensitive compares)
    return F.lower(F.trim(F.coalesce(F.col(c).cast("string"), F.lit(""))))

def _raw(c):      # trim, null -> ""         (case preserved)
    return F.trim(F.coalesce(F.col(c).cast("string"), F.lit("")))

def _notblank(c):
    return F.col(c).isNotNull() & (F.trim(F.col(c).cast("string")) != "")

def _clean2(c):   # trimmed value, or NULL when blank (for COALESCE fallbacks)
    t = F.trim(F.col(c).cast("string"))
    return F.when(t.isNotNull() & (t != ""), t)

def _has(colexpr, subs):
    e = F.lit(False)
    for s in subs:
        e = e | colexpr.contains(s)
    return e

def _in(colexpr, vals):     # colexpr already lowered; vals lowercased here
    return colexpr.isin(*[v.lower() for v in vals])

# --- lowered base expressions (reused) --------------------------------------
appHost   = _s("AppHost")
ctxType   = _s("Context_Type")
resType   = _s("AccessedResource_Type")
resAction = _s("AccessedResource_Action")
siteUrl   = _s("AccessedResource_SiteUrl")
pluginId  = _s("AISystemPlugin_Id")
isActive  = _has(resAction, ["send","draft","create","post","invoke","write","patch","execute"])
HYPER     = "http://schema.skype.com/hyperlink"

# --- Environment / License Status -------------------------------------------
hl    = F.upper(F.trim(F.coalesce(F.col("Has license").cast("string"), F.lit(""))))
isLic = hl.isin("YES","TRUE","Y","1")
fact = fact.withColumn("Environment", F.when(isLic, F.lit("Licensed")).otherwise(F.lit("Unlicensed")))
fact = fact.withColumn("License Status",
                       F.when(isLic, F.lit("M365 Copilot Licensed")).otherwise(F.lit("Unlicensed")))
env = F.col("Environment")
actor_env = chain([
    (appHost.contains("cowork"), "Cowork"),
    (_notblank("AgentName") | _notblank("AgentId") | appHost.isin("autonomous", "logic app") | resType.isin("flow", "connector"), "Agents"),
], "User")

# --- Is_Sensitive -----------------------------------------------------------
fact = fact.withColumn("Is_Sensitive",
                       (_notblank("SensitivityLabelId") | _notblank("AccessedResource_SensitivityLabelId")))

# --- AI_Model ---------------------------------------------------------------
mdl = F.upper(F.coalesce(F.col("ModelTransparencyDetails_ModelName").cast("string"), F.lit("")))
fact = fact.withColumn("AI_Model", chain([
    ((mdl == "") | (mdl == "NULL"), "Embedded App (no model logged)"),
    (mdl.contains("DEEP_LEO"), "GPT-4 (Standard)"),
    (mdl.contains("REASONING"), "Reasoning Model (o1/o3)"),
    (mdl.contains("OFFENSIVE"), "Safety Filter (blocked)"),
    (mdl.contains("GPT-41") | mdl.contains("GPT-4.1"), "GPT-4.1 (Next Gen)"),
    (mdl.contains("O3-MINI") | mdl.contains("O3MINI"), "o3-mini (Reasoning)"),
    (mdl.contains("O3") | mdl.contains("O1"), "Reasoning Model (o-series)"),
    (mdl.contains("GPT-5") | mdl.contains("GPT5"), "GPT-5 (Next Gen)"),
    (mdl.contains("CLAUDE"), "Claude (Anthropic)"),
    (mdl.contains("GEMINI"), "Gemini (Google)"),
    (mdl.contains("LLAMA") | mdl.contains("META"), "LLaMA (Meta)"),
    (mdl.contains("PHI"), "Phi (Microsoft Small Model)"),
], mdl))

# --- Behavior_Category ------------------------------------------------------
from_resource = chain([
    (_in(resAction, ["sendemailv2","draftemail","senddraftemail","updatedraftemail"]), "Email Drafting"),
    ((resType == "emailmessage") & isActive, "Email Drafting"),
    (resType == "emailmessage", "Email Summarising"),
    (resAction == "mcp_meetingmanagement", "Meeting Scheduling"),
    (_in(resType, ["event","teamsmeeting"]), "Meeting Prep"),
    (_in(resAction, ["postmessagetoconversation","createchat"]), "Teams Messaging"),
    (_in(resType, ["teamsmessage","teamschat","teamschannel"]), "Teams Messaging"),
    (resType == "flow", "Running a Workflow"),
    (_in(resType, ["connector","http"]) & isActive, "Running a Workflow"),
    (_in(resAction, ["executedatasetquery","getitems","getalltables","gettableviews"]), "Data Querying"),
    (_in(resType, ["xlsx","csv","xlsm","xlsb","xls"]),
        F.when(isActive, F.lit("Excel Assistance")).otherwise(F.lit("Spreadsheet Review"))),
    (resType == "peopleinferenceanswer", "People Lookup"),
    ((resType == "listitem") | (resType == "aspx"), "Enterprise Searching"),
    (resType == "websearchquery", "Web Searching"),
    (resType == "pdf", "PDF Analysis"),
    (_in(resType, ["py","js","java","tsx","jsx","css","php","sh"]) & isActive, "Code Writing"),
    (_in(resType, ["py","sql","js","java","json","xml","html","yaml","yml","txt"]), "Code Analysis"),
    (_in(resType, ["png","jpg","jpeg","svg","gif"]) & isActive, "Image Generation"),
    (_in(resType, ["png","jpg","jpeg","gif"]), "Image / Media Analysis"),
    (_in(resType, ["streamvideo","mp4","mov","webm","mkv"]), "Video Summarising"),
    (_in(resType, ["planid","taskids"]), "Task Management"),
    (resType == "looppage", "Real-time Collaboration"),
    ((resType == HYPER) & _has(siteUrl, ["github.com","stackoverflow.com","npmjs.com","pypi.org","docker.com","kubernetes.io","leetcode.com"]), "Code Analysis"),
    ((resType == HYPER) & _has(siteUrl, ["learning.cloud.microsoft","coursera.org","udemy.com"]), "Coaching"),
    ((resType == HYPER) & siteUrl.contains("sharepoint.com"), "Enterprise Searching"),
    (_in(resType, ["external","http"]) | (resType == HYPER), "Web Searching"),
    (_in(resType, ["docx","doc","rtf"]) & isActive, "Document Drafting"),
    (_in(resType, ["docx","doc","rtf"]) & (resAction == "read"), "File Retrieval"),
    (_in(resType, ["docx","doc","rtf"]), "Document Summarising"),
    (_in(resType, ["pptx","ppt","potx"]) & isActive, "Presentation Creation"),
    (_in(resType, ["pptx","ppt","potx"]) & (resAction == "read"), "File Retrieval"),
    (_in(resType, ["pptx","ppt","potx"]), "Presentation Summarising"),
    (_has(siteUrl, ["service-now.com","servicenow.com"]), "IT & Service Desk"),
    (siteUrl.contains("dynamics.com"), "Sales & Customer"),
], None)

from_plugin = F.when(pluginId == "enterprisesearch", F.lit("Enterprise Searching")).otherwise(F.lit(None).cast("string"))

from_context = chain([
    (ctxType == "teamsmeeting", "Meeting Prep"),
    (ctxType == "streamvideo", "Video Summarising"),
    (ctxType == "docx",
        F.when((appHost == "word") & isActive, F.lit("Document Drafting")).otherwise(F.lit("Document Summarising"))),
    (_in(ctxType, ["xlsx","xlsm","xlsb","xls","csv"]), "Spreadsheet Review"),
    (_in(ctxType, ["pptx","pptm"]),
        F.when((appHost == "powerpoint") & isActive, F.lit("Presentation Creation")).otherwise(F.lit("Presentation Summarising"))),
    (_in(ctxType, ["teamschat","teamschannel"]), "Teams Messaging"),
    (ctxType == "aspx", "Enterprise Searching"),
    ((appHost.isin("outlook","outlooksidepane")) & isActive, "Email Drafting"),
    (appHost.isin("outlook","outlooksidepane"), "Email Summarising"),
    (appHost == "excel", "Excel Assistance"),
    ((appHost == "word") & isActive, "Document Drafting"),
    (appHost == "word", "Document Summarising"),
    ((appHost == "powerpoint") & isActive, "Presentation Creation"),
    (appHost == "powerpoint", "Presentation Summarising"),
    (appHost == "stream", "Video Summarising"),
    (appHost == "sharepoint", "SharePoint Access"),
    (appHost == "designer", "Image Generation"),
    (appHost == "onenote", "Note Taking"),
    (appHost == "forms", "Form / Survey Work"),
    (appHost == "planner", "Task Management"),
    (_in(appHost, ["loop","whiteboard","vivaengage"]), "Real-time Collaboration"),
    (appHost == "copilot studio", "Domain-Specific Agent"),
    (appHost == "autonomous", "Running a Workflow"),
    ((appHost == "logic app") & (_notblank("AgentName") | _notblank("AgentId")), "Running a Workflow"),
    (_in(appHost, ["datawarehousing core","power bi"]), "Data Querying"),
], "General Chat")

fact = fact.withColumn("Behavior_Category", F.coalesce(from_resource, from_plugin, from_context))

# --- Behavior_Enriched ------------------------------------------------------
agentName_l = _s("AgentName")
bcat = F.col("Behavior_Category")
qna_set = ["General Q&A","M365 Chat Q&A","Teams Q&A","Browser Q&A","General Chat"]
fact = fact.withColumn("Behavior_Enriched", chain([
    ((actor_env != "Agents") & (actor_env != "Cowork"), bcat),
    (~bcat.isin(*qna_set), bcat),
    (_has(agentName_l, ["coach","mentor","learning","career"]), "Coaching"),
    (_has(agentName_l, ["research","analyst","analy"]), "Research & Analysis"),
    (_has(agentName_l, ["sales","commercial","customer","crm","revenue"]), "Sales & Customer"),
    (_has(agentName_l, ["hr","recruit","talent","onboard","people"]), "HR & People"),
    (_has(agentName_l, ["policy","compliance","legal","audit","risk"]), "Compliance & Policy"),
    (_has(agentName_l, ["service","support","help","ticket","incident"]), "IT & Service Desk"),
    (_has(agentName_l, ["summar","draft","translat","editor"]), "Content Generation"),
    (_has(agentName_l, ["data","report","dashboard","metric"]), "Data & Reporting"),
    (_has(agentName_l, ["knowledge","faq","wiki","buddy","guide"]), "Knowledge Base"),
    (_has(agentName_l, ["idea","brainstorm","creative","design"]), "Ideation & Creative"),
], "General Assistance"))

# --- Agents 365 lookup (Agent_LinkID -> Title ID) for the *_Full enrichment --
def _a365_lookup():
    try:
        a = spark.table(SRC_AGENTS)
    except Exception as e:
        print(f"[warn] {SRC_AGENTS} not found ({e}); A365 enrichment degrades to blank.")
        return None
    tid = first_existing(a, ["Title ID","Title Id","TitleID","Title ID "])
    if not tid:
        return None
    def pick(names):
        c = first_existing(a, names)
        return F.col(c).cast("string") if c else F.lit(None).cast("string")
    out = (a.withColumn("_a_title", F.trim(F.col(tid).cast("string")))
             .where(F.col("_a_title").isNotNull() & (F.col("_a_title") != ""))
             .withColumn("A365_Desc",   pick(["Agent description","Agent description ","Agent Description"]))
             .withColumn("A365_Actions",pick(["Custom actions","Custom actions ","Custom Actions"]))
             .withColumn("A365_Code",   pick(["Can use code interpreter","Can use code interpreter "]))
             .withColumn("A365_Images", pick(["Can generate images using user prompt","Can generate images using user prompt "]))
             .withColumn("A365_SP",     pick(["Can read Sharepoint sites and files","Can read Sharepoint sites and files "]))
             .withColumn("A365_Type",   pick(["Agent type (A365)","Agent type (A365) ","Agent Type (A365)"]))
             .select("_a_title","A365_Desc","A365_Actions","A365_Code","A365_Images","A365_SP","A365_Type")
             .dropDuplicates(["_a_title"]))
    return out

_a365 = _a365_lookup()
if _a365 is not None:
    fact = fact.join(F.broadcast(_a365), fact["Agent_LinkID"] == _a365["_a_title"], "left").drop("_a_title")
for _c in ["A365_Desc","A365_Actions","A365_Code","A365_Images","A365_SP","A365_Type"]:
    fact = ensure_col(fact, _c)

# --- Behavior_Enriched_Full -------------------------------------------------
be   = F.col("Behavior_Enriched")
needs = (actor_env == "Agents") & (be == "General Assistance")
desc    = F.when(needs, F.lower(F.coalesce(F.col("A365_Desc").cast("string"), F.lit("")))).otherwise(F.lit(""))
actions = F.when(needs, F.lower(F.coalesce(F.col("A365_Actions").cast("string"), F.lit("")))).otherwise(F.lit(""))
search  = F.concat_ws(" ", desc, actions)
hasCode   = needs & (F.col("A365_Code")   == "Yes")
hasImages = needs & (F.col("A365_Images") == "Yes")
hasSP     = needs & (F.col("A365_SP")     == "Yes")
resolved = chain([
    (~needs, be),
    (_has(search, ["coach","mentor","learning","training","skill"]), "Coaching"),
    (_has(search, ["research","analyst","analy","insight","intelligence"]), "Research & Analysis"),
    (_has(search, ["sales","commercial","customer","crm","pipeline","prospect","deal"]), "Sales & Customer"),
    (_has(search, ["recruit","talent","onboard","hiring","employee","human resource","job description"]), "HR & People"),
    (_has(search, ["policy","compliance","legal","audit","risk","governance"]), "Compliance & Policy"),
    (_has(search, ["support","helpdesk","troubleshoot","ticket","incident","service desk"]), "IT & Service Desk"),
    (_has(search, ["summar","draft","translat","content","communications"]), "Content Generation"),
    (_has(search, ["data","report","dashboard","analytics","metric"]), "Data & Reporting"),
    (_has(search, ["knowledge","faq","wiki","guide","handbook","documentation"]), "Knowledge Base"),
    (_has(search, ["brainstorm","creative","design","innovat"]), "Ideation & Creative"),
    (hasCode, "Data & Reporting"),
    (hasImages, "Ideation & Creative"),
    (hasSP, "Knowledge Base"),
], "General Assistance")
fact = fact.withColumn("_Resolved", resolved)
wfsig = F.lower(F.concat_ws(" | ",
    F.coalesce(F.col("AgentName").cast("string"), F.lit("")),
    F.coalesce(F.col("AccessedResource_SiteUrl").cast("string"), F.lit("")),
    F.coalesce(F.col("AccessedResource_Action").cast("string"), F.lit("")),
    F.coalesce(F.col("AppHost").cast("string"), F.lit(""))))
fact = fact.withColumn("Behavior_Enriched_Full", chain([
    (F.col("_Resolved") != "Running a Workflow", F.col("_Resolved")),
    (_has(wfsig, ["servicenow","salesforce","dynamics","workday","jira","zendesk","service desk","servicedesk"]), "Specialist / Line-of-Business Workflow"),
    (_has(wfsig, ["outlook","exchange","mail"]), "Email Workflow"),
    (_has(wfsig, ["calendar","meeting","schedul"]), "Meeting Workflow"),
    (_has(wfsig, ["power bi","powerbi","dataverse","dataset","report","dashboard","excel","sql","analytics"]), "Data & Reporting Workflow"),
    (_has(wfsig, ["sharepoint","onedrive","word","document",".doc","file"]), "Document Workflow"),
    (_has(wfsig, ["planner","task","approv","teams","notify","post","list"]), "Coordination Workflow"),
], "General Workflow")).drop("_Resolved")

# --- Behavior_Source --------------------------------------------------------
agent_r  = _raw("AgentName")
plugin_r = _raw("AISystemPlugin_Name")
app_r    = _raw("AppHost")
src_expr = chain([
    (actor_env == "Cowork", F.concat(F.lit("Cowork"),
        F.when(agent_r != "", F.concat(F.lit(": "), agent_r)).otherwise(F.lit("")))),
    ((actor_env == "Agents") & (agent_r != ""), F.concat(F.lit("Agent: "), agent_r)),
    (plugin_r != "", F.concat(app_r, F.lit(" ("), plugin_r, F.lit(")"))),
    (app_r != "", app_r),
], "Copilot Chat")
fact = fact.withColumn("Behavior_Source", F.concat(bcat, F.lit(" -> "), src_expr))

# --- Value_Outcome ----------------------------------------------------------
isSens = F.col("Is_Sensitive")
fact = fact.withColumn("Value_Outcome", chain([
    (be.isin("Email Summarising","Email Triage","Email Thread Summary"), "Time Saved (Email)"),
    (be.isin("Meeting Prep","Video Summarising"), "Time Saved (Meetings)"),
    (be.isin("Document Summarising","Presentation Summarising","Note Taking"), "Time Saved (Documents)"),
    (be.isin("Web Searching","Enterprise Searching","File Retrieval","PDF Analysis","SharePoint Access","People Lookup","Knowledge Base"), "Search Time Saved"),
    (be.isin("Teams Messaging","Meeting Scheduling"), "Communication Time Saved"),
    (be.isin("Spreadsheet Review","Spreadsheet Analysis","Excel Assistance"), "Spreadsheet Time Saved"),
    (be.isin("Email Drafting","Document Drafting","Presentation Creation","Image Generation","Image / Media Analysis","Image/Media Analysis","Content Generation","Ideation & Creative"), "Content Output"),
    (be.isin("Real-time Collaboration","Form / Survey Work"), "Team Collaboration"),
    ((be == "Running a Workflow") | (actor_env == "Cowork"), "Workflow Automation"),
    (be == "Task Management", "Task Coordination"),
    (isSens & (actor_env != "Agents") & (actor_env != "Cowork"), "Compliance & Risk"),
    (be.isin("Data Querying","Data & Reporting","Research & Analysis"), "Data-Driven Decisions"),
    (be.isin("Code Writing","Code Analysis","Code Analysis (URL)"), "Coding Capability"),
    (be.isin("Coaching","Coaching (URL)"), "Skills Development"),
    (be == "Sales & Customer", "Revenue Enablement"),
    (be == "IT & Service Desk", "Service Desk Deflection"),
    (be == "Compliance & Policy", "Compliance & Risk"),
    (be == "HR & People", "HR Expertise"),
    (be.isin("Domain-Specific Agent","Cross-Org Agent"), "Specialist Expertise"),
], "General AI Productivity"))

# --- Usage_Mode -------------------------------------------------------------
bef = F.col("Behavior_Enriched_Full")
a365type = F.coalesce(F.col("A365_Type").cast("string"), F.lit(""))
isDelegating = (actor_env == "Cowork") | (appHost == "autonomous") | bef.contains("Workflow") | a365type.isin("Autonomous","Workflow","Triggered")
producing = ["Email Drafting","Document Drafting","Presentation Creation","Image Generation","Code Writing","Code Analysis","Code Analysis (URL)","Data Querying","Spreadsheet Analysis","Excel Assistance","Content Generation","Ideation & Creative","Research & Analysis","Data & Reporting","Sales & Customer","HR & People","IT & Service Desk","Compliance & Policy","Coaching","Coaching (URL)","Domain-Specific Agent","Cross-Org Agent","Form / Survey Work","Real-time Collaboration","Note Taking","Teams Messaging","Meeting Scheduling","Task Management"]
consuming = ["Document Summarising","Email Summarising","Email Thread Summary","Email Triage","Presentation Summarising","Video Summarising","Meeting Prep","Image / Media Analysis","Image/Media Analysis","Sensitive Content Interaction"]
finding   = ["Web Searching","Enterprise Searching","PDF Analysis","SharePoint Access","File Retrieval","People Lookup","Knowledge Base","Spreadsheet Review"]
fact = fact.withColumn("Usage_Mode", chain([
    (isDelegating, "5 - Delegating"),
    (bef.isin(*producing), "4 - Producing"),
    (bef.isin(*consuming), "3 - Consuming"),
    (bef.isin(*finding), "2 - Finding"),
], "1 - Asking"))

# --- Expertise_Role ---------------------------------------------------------
fact = fact.withColumn("Expertise_Role", chain([
    (bef.isin("Data Querying","Data & Reporting","Spreadsheet Analysis"), "Data Analyst"),
    (bef.isin("Code Writing","Code Analysis","Code Analysis (URL)"), "Software Engineer"),
    (bef.isin("Research & Analysis"), "Business Analyst"),
    (bef.isin("Compliance & Policy","Sensitive Content Interaction"), "Compliance Specialist"),
    (bef.isin("Sales & Customer"), "Sales Consultant"),
    (bef.isin("IT & Service Desk"), "IT Specialist"),
    (bef.isin("HR & People"), "HR Specialist"),
    (bef.isin("Coaching","Coaching (URL)"), "Coach"),
    (bef.contains("Workflow") | (bef == "Task Management"), "Automation Engineer"),
    (bef.isin("Domain-Specific Agent","Cross-Org Agent"), "Domain Expert"),
    (bef.isin("Email Drafting"), "Communications Specialist"),
    (bef.isin("Email Triage","Meeting Scheduling","Email Summarising","Email Thread Summary"), "Executive Assistant"),
    (bef.isin("Document Drafting","Content Generation","Note Taking","Document Summarising"), "Content Writer"),
    (bef.isin("Presentation Creation","Presentation Summarising"), "Presentation Designer"),
    (bef.isin("Image Generation","Image/Media Analysis","Image / Media Analysis","Ideation & Creative"), "Visual Designer"),
    (bef.isin("Meeting Prep","Video Summarising"), "Meeting Coordinator"),
    (bef.isin("Web Searching","PDF Analysis","Knowledge Base"), "Researcher"),
    (bef.isin("Enterprise Searching","SharePoint Access","File Retrieval","People Lookup"), "Knowledge Navigator"),
    (bef.isin("Spreadsheet Review","Excel Assistance"), "Spreadsheet Specialist"),
    (bef.isin("Real-time Collaboration","Form / Survey Work","Form/Survey Work","Teams Messaging"), "Collaboration Lead"),
], None))

# --- Efficiency_Breakdown ---------------------------------------------------
fact = fact.withColumn("Efficiency_Breakdown", chain([
    (bef.isin("Email Summarising","Email Triage","Email Thread Summary","Email Drafting"), "Email"),
    (bef.isin("Document Summarising","Note Taking","Document Drafting","Content Generation"), "Document Assistance"),
    (bef.isin("Presentation Summarising","Presentation Creation"), "Presentations"),
    (bef.isin("Meeting Prep","Video Summarising","Meeting Scheduling"), "Meetings"),
    (bef.isin("Web Searching","Enterprise Searching","PDF Analysis","SharePoint Access","File Retrieval","People Lookup","Knowledge Base","Research & Analysis"), "Search & Research"),
    (bef.isin("Spreadsheet Review","Excel Assistance","Spreadsheet Analysis","Data Querying","Data & Reporting"), "Data & Spreadsheets"),
    (bef.isin("Image Generation","Image / Media Analysis","Image/Media Analysis","Ideation & Creative","Code Writing","Code Analysis","Code Analysis (URL)"), "Creative & Technical"),
    (bef.isin("Teams Messaging","Real-time Collaboration","Form / Survey Work","Task Management") | bef.contains("Workflow"), "Collaboration & Workflows"),
    (bef.isin("Sales & Customer","IT & Service Desk","HR & People","Compliance & Policy","Coaching","Coaching (URL)","Domain-Specific Agent","Cross-Org Agent"), "Specialist Support"),
    (bcat == "Teams Q&A", "Teams Chat"),
    (bcat == "M365 Chat Q&A", "BizChat Q&A"),
    (bcat == "Browser Q&A", "BizChat Q&A"),
], "General Q&A"))

# --- Web_Grounded_Signal ----------------------------------------------------
isInternal = _has(siteUrl, ["sharepoint.com",".onmicrosoft.com"])
fact = fact.withColumn("Web_Grounded_Signal",
    F.when((resType == "websearchquery") | _in(resType, ["external","http"]) | ((resType == HYPER) & ~isInternal),
           F.lit("Web Grounded")).otherwise(F.lit("Not Web Grounded")))

# --- Behavior_Plausible -----------------------------------------------------
lic = F.col("License Status")
unlic_ok = ["General Chat","Web Searching","PDF Analysis","Document Summarising","Image / Media Analysis","Image Generation","Code Analysis","Translation"]
plausible_switch = chain([
    (bcat.isin("Email Summarising","Email Drafting"), "Free Chat Workaround (pasting Email)"),
    (bcat.isin("Excel Assistance","Spreadsheet Review","Data Querying"), "Free Chat Workaround (pasting Spreadsheet/Data)"),
    (bcat.isin("Meeting Prep","Meeting Scheduling"), "Free Chat Workaround (pasting Meeting info)"),
    (bcat == "Teams Messaging", "Free Chat Workaround (pasting Teams content)"),
    (bcat.isin("Enterprise Searching","People Lookup"), "Free Chat Workaround (pasting Enterprise data)"),
    (bcat.isin("Running a Workflow","Task Management"), "Free Chat Workaround (pasting Workflow)"),
    (bcat == "Real-time Collaboration", "Free Chat Workaround (pasting Loop content)"),
    (bcat == "Code Writing", "Free Chat Workaround (pasting Code)"),
    (bcat == "Video Summarising", "Free Chat Workaround (uploading Video)"),
], "Free Chat Workaround (Other)")
fact = fact.withColumn("Behavior_Plausible",
    F.when((lic == "M365 Copilot Licensed") | bcat.isin(*unlic_ok), bcat).otherwise(plausible_switch))

# --- Workflow_Action --------------------------------------------------------
fact = fact.withColumn("Workflow_Action",
    F.when(~bef.contains("Workflow"), F.lit("")).otherwise(chain([
        (_has(resAction, ["send","post","notify"]), "Sending / Notifying"),
        (_has(resAction, ["create","draft","write","add"]), "Creating Content"),
        (_has(resAction, ["invoke","execute","trigger","run"]), "Invoking / Triggering"),
        (_has(resAction, ["update","patch","modify","set"]), "Updating Records"),
        (_has(resAction, ["read","get","list","fetch"]), "Reading Data"),
        (_has(resAction, ["delete","remove"]), "Deleting / Removing"),
        (appHost == "autonomous", "Autonomous Run (no action logged)"),
        (appHost == "logic app", "Logic App Run (no action logged)"),
    ], "Workflow (other)")))

# --- Is_Agent_Activity ------------------------------------------------------
isAutonomous = appHost.isin("autonomous","logic app") | resType.isin("flow","connector")
fact = fact.withColumn("Is_Agent_Activity", (_notblank("AgentName") | _notblank("AgentId") | isAutonomous))

# --- Agent Filter -----------------------------------------------------------
isCowork = actor_env == "Cowork"
fact = fact.withColumn("Agent Filter",
    F.when(isCowork, F.lit("Cowork"))
     .when(F.col("Is_Agent_Activity") == True, F.lit("Agents"))
     .otherwise(F.lit(None).cast("string")))

# --- Grounding Source -------------------------------------------------------
plugin = F.lower(F.concat_ws(" ",
    F.coalesce(F.col("AISystemPlugin_Id").cast("string"), F.lit("")),
    F.coalesce(F.col("AISystemPlugin_Name").cast("string"), F.lit(""))))
usedWeb = _has(plugin, ["bing","web"])
usedInternal = (F.col("Resource_Count") > 0) | _notblank("Context_Type")
fact = fact.withColumn("Grounding Source", chain([
    (usedWeb & usedInternal, "Mixed (Internal+Web)"),
    (usedWeb, "External (Web)"),
    (usedInternal, "Internal"),
], "Ungrounded"))

# --- Agent_Surface ----------------------------------------------------------
host = appHost
agent_l = agentName_l
fact = fact.withColumn("Agent_Surface", chain([
    (host.contains("cowork") | agent_l.contains("cowork"), "Cowork"),
    (agent_l.contains("scout"), "Scout"),
    (host.isin("autonomous","logic app") | resType.isin("flow","connector"), "Autonomous / Flow"),
    (F.col("Is_Agent_Activity") == True, "Copilot Agents"),
], None))

# --- Agent type / publisher / consolidated name -----------------------------
# Rules live in describe_agent (helpers cell). Each distinct combination of agent
# fields is classified once on the driver, then broadcast-joined back (no UDF).
AGENT_TYPE_COLS = ["Agent_Key", "Agent_Type", "Agent_Type_Basis", "Agent_Publisher",
                   "Agent_Is_Published", "Agent_Consolidated_Name"]
_at_inputs = [
    ("__at_aid", F.coalesce(_clean2("Agent_TargetPlatformId"), _clean2("AgentId"), _clean2("PlatformAgentId"))),
    ("__at_name", F.coalesce(_clean2("Agent_TargetName"), _clean2("AgentName"))),
    ("__at_app", _clean2("AppIdentity_Text")),
    ("__at_type", F.coalesce(_clean2("Agent_PlatformType"), _clean2("AgentPlatform"))),
    ("__at_workload", _clean2("Workload")),
]
_at_keys = [name for name, _ in _at_inputs]
fact = fact.drop(*AGENT_TYPE_COLS)
for _name, _expr in _at_inputs:
    fact = fact.withColumn(_name, F.coalesce(_expr, F.lit("")))

_at_overrides = {}
if AGENT_TYPE_OVERRIDES_TABLE and spark.catalog.tableExists(AGENT_TYPE_OVERRIDES_TABLE):
    _ov = spark.table(AGENT_TYPE_OVERRIDES_TABLE)
    _ov_cols = {c.lower(): c for c in _ov.columns}
    _ov_key, _ov_type = _ov_cols.get("key"), _ov_cols.get("agent_type") or _ov_cols.get("category")
    if _ov_key and _ov_type:
        _at_overrides = build_agent_type_overrides(_ov.select(_ov_key, _ov_type).collect())
        print(f"agent type: {len(_at_overrides):,} overrides from {AGENT_TYPE_OVERRIDES_TABLE}")
    else:
        print(f"agent type: {AGENT_TYPE_OVERRIDES_TABLE} needs 'key' and 'Agent_Type' columns; ignored")

_at_rows = [tuple(r) + describe_agent(*r, overrides=_at_overrides)
            for r in fact.select(*_at_keys).distinct().collect()]
_at_schema = T.StructType(
    [T.StructField(name, T.StringType()) for name in _at_keys]
    + [T.StructField(name, T.BooleanType() if name == "Agent_Is_Published" else T.StringType())
       for name in AGENT_TYPE_COLS])
fact = fact.join(F.broadcast(spark.createDataFrame(_at_rows, _at_schema)), _at_keys, "left").drop(*_at_keys)
_at_types = {}
for _r in _at_rows:
    _at_types[_r[len(_at_keys) + 1]] = _at_types.get(_r[len(_at_keys) + 1], 0) + 1
print("agent type: key combinations by type: "
      + ", ".join(f"{k or 'not an agent'}={v:,}" for k, v in sorted(_at_types.items(), key=lambda kv: str(kv[0]))))

# --- Execution_Trigger ------------------------------------------------------
fact = fact.withColumn("Execution_Trigger",
    F.when(host.isin("autonomous","logic app") | resType.isin("flow","connector"),
           F.lit("Scheduled / Autonomous")).otherwise(F.lit("Interactive / On-demand")))

# --- UserMonthKey / Delegation_Event_Key / ActivityDate ---------------------
fact = fact.withColumn("UserMonthKey",
    F.concat(F.coalesce(F.col("Audit_UserId").cast("string"), F.lit("")), F.lit("|"),
             F.date_format(F.col("MonthStart"), "yyyy-MM")))
fact = fact.withColumn("Delegation_Event_Key",
    F.concat(F.coalesce(F.col("Audit_UserId").cast("string"), F.lit("")), F.lit("|"),
             F.date_format(F.col("InteractionDate"), "yyyy-MM-dd"), F.lit("|"),
             F.coalesce(_clean2("AgentName"), _clean2("Workflow_Action"), _clean2("AppHost"), F.lit("unknown-workflow"))))
fact = fact.withColumn("ActivityDate", F.col("InteractionDate").cast("timestamp"))

# --- Agent Last Used Date  (per-AgentName max CreationDate) ------------------
# Aggregate then broadcast-join: a window partitioned by AgentName sends every row
# without an agent (most of them) to one task. KEPT_AGENT_LAST_USED adds the months an
# incremental run does not recompute, so the value matches a full rebuild.
_alu = (fact.where(_notblank("AgentName")).groupBy("AgentName")
        .agg(F.max("CreationDate").alias("__alu")))
if globals().get("KEPT_AGENT_LAST_USED"):
    _alu = (_alu.unionByName(spark.createDataFrame(
                KEPT_AGENT_LAST_USED, T.StructType([T.StructField("AgentName", fact.schema["AgentName"].dataType),
                                                    T.StructField("__alu", T.TimestampType())])))
            .groupBy("AgentName").agg(F.max("__alu").alias("__alu")))
_alu = _alu.withColumnRenamed("AgentName", "__alu_name")
_alu_cols = fact.columns if "Agent Last Used Date" in fact.columns else fact.columns + ["Agent Last Used Date"]
fact = (fact.join(F.broadcast(_alu), fact["AgentName"] == _alu["__alu_name"], "left")
        .withColumn("Agent Last Used Date",
                    F.when(_notblank("AgentName"), F.col("__alu")).otherwise(F.lit(None).cast("timestamp")))
        .select(*[F.col(f"`{c}`") for c in _alu_cols]))

# --- User_Stage_Maturity / User_Stage  (UserMonthMetrics staging) -----------
umm = (fact.groupBy("Audit_UserId", "MonthStart")
    .agg(
        F.countDistinct("Behavior_Enriched_Full").alias("BehaviorCount"),
        F.max(F.when(_notblank("AgentName"), F.lit(1)).otherwise(F.lit(0))).alias("_HasAgent"),
        F.countDistinct("InteractionDate").alias("ActiveDays"),
        F.sum(F.when(F.col("Usage_Mode").isin("4 - Producing","5 - Delegating"), F.lit(1)).otherwise(F.lit(0))).alias("_ValueRows"),
        F.count(F.lit(1)).alias("_TotalRows"),
    ))
umm = umm.withColumn("HasAgent", F.col("_HasAgent") > 0)
umm = umm.withColumn("ValueFocusShare",
    F.when(F.col("_TotalRows") > 0, F.col("_ValueRows") / F.col("_TotalRows")).otherwise(F.lit(0.0)))
umm = umm.withColumn("UserStage", chain([
    (((F.col("ActiveDays") >= 15) | ((F.col("ActiveDays") >= 10) & (F.col("ValueFocusShare") >= 0.30) & F.col("HasAgent"))), "4 - Power"),
    ((F.col("ActiveDays") >= 8) & (F.col("BehaviorCount") >= 5), "3 - Habitual"),
    ((F.col("ActiveDays") >= 3) & (F.col("BehaviorCount") >= 3), "2 - Developing"),
], "1 - Beginner"))
umm = umm.withColumn("_UMKey",
    F.concat(F.coalesce(F.col("Audit_UserId").cast("string"), F.lit("")), F.lit("|"),
             F.date_format(F.col("MonthStart"), "yyyy-MM"))).select("_UMKey", "UserStage")
fact = fact.join(F.broadcast(umm), fact["UserMonthKey"] == umm["_UMKey"], "left").drop("_UMKey")
fact = fact.withColumnRenamed("UserStage", "User_Stage_Maturity")
fact = fact.withColumn("User_Stage", F.col("User_Stage_Maturity"))

# --- drop A365 helper columns (not part of the model contract) --------------
fact = fact.drop("A365_Desc","A365_Actions","A365_Code","A365_Images","A365_SP","A365_Type")

ENRICHED_COLS = ["Environment","License Status","Is_Sensitive","AI_Model","Behavior_Category",
    "Behavior_Enriched","Behavior_Enriched_Full","Behavior_Source","Value_Outcome","Usage_Mode",
    "Expertise_Role","Efficiency_Breakdown","Web_Grounded_Signal","Behavior_Plausible","Workflow_Action",
    "Is_Agent_Activity","Agent Filter","Grounding Source","Agent_Surface","Execution_Trigger",
    "UserMonthKey","Delegation_Event_Key","ActivityDate","Agent Last Used Date",
    "User_Stage_Maturity","User_Stage"] + AGENT_TYPE_COLS
_missing_enr = [c for c in ENRICHED_COLS if c not in fact.columns]
assert not _missing_enr, f"enrichment incomplete, missing: {_missing_enr}"
print(f"enrichment complete: +{len(ENRICHED_COLS)} columns (fact now {len(fact.columns)} cols)")


In [ ]:
# ----------------------------------------------------------------------------
# 11. WRITE CURATED DELTA  (Power BI reads this table verbatim)
#     Stored by MonthStart. A full rebuild overwrites the table; an incremental run
#     replaces only the recomputed months (replaceWhere) and leaves the rest as is.
#     The per-month source fingerprints are saved as a table property for the next run.
# ----------------------------------------------------------------------------
def save_processor_state(state_json):
    _value = (state_json or "").replace("\\", "\\\\").replace("'", "\\'")
    spark.sql(f"ALTER TABLE {OUT_TABLE} SET TBLPROPERTIES ('{PROCESSOR_STATE_PROPERTY}' = '{_value}')")

def write_curated_output(df: DataFrame, plan) -> str:
    if plan["full"]:
        (df.write.mode("overwrite")
            .option("overwriteSchema", "true")
            .option("delta.columnMapping.mode", "name")
            .option("delta.minReaderVersion", "2")
            .option("delta.minWriterVersion", "5")
            .partitionBy(PARTITION_COLUMN)
            .format("delta").saveAsTable(OUT_TABLE))
        return "overwrite"
    if not plan["months"]:
        return "unchanged"
    target_columns = spark.table(OUT_TABLE).columns
    if sorted(df.columns) != sorted(target_columns):
        # Settings, logic and source schema are fingerprinted, so this should not happen.
        # Clear the state so the next run rebuilds the whole table.
        save_processor_state("")
        raise RuntimeError(
            f"{OUT_TABLE} columns differ from this run's output "
            f"(missing {sorted(set(target_columns) - set(df.columns))}, "
            f"new {sorted(set(df.columns) - set(target_columns))}). "
            "The next run rebuilds the whole table, or run WRITE_MODE='overwrite' now.")
    # Only the recomputed months are replaced; each month is rebuilt from all its source rows.
    (df.select(*[F.col(f"`{c}`") for c in target_columns]).write.mode("overwrite")
        .option("replaceWhere", month_predicate_sql(PARTITION_COLUMN, plan["months"]))
        .format("delta").saveAsTable(OUT_TABLE))
    return "incremental"

if INCLUDE_RAW_PASSTHROUGH:
    validate_passthrough_columns(passthrough_source_columns, fact.columns)
    print("raw passthrough: source column presence guard passed (canonical transformations unchanged)")

print(f"curated: {len(fact.columns)} cols -> {OUT_TABLE}  (requested={WRITE_MODE}, "
      f"{'full rebuild' if PROCESS_PLAN['full'] else 'incremental'}, {len(PROCESS_PLAN['months'])} month(s))")
write_strategy = write_curated_output(fact, PROCESS_PLAN)
save_processor_state(build_processor_state(PROCESSOR_CONFIG, SOURCE_MONTHS))

if RUN_OPTIMIZE and write_strategy != "unchanged":
    if write_strategy == "overwrite":
        spark.sql(f"OPTIMIZE {OUT_TABLE} VORDER")   # compact + V-Order for Direct Lake / fast import
    else:
        spark.sql(f"OPTIMIZE {OUT_TABLE} WHERE {month_predicate_sql(PARTITION_COLUMN, PROCESS_PLAN['months'])} VORDER")

print("done. row count:", spark.table(OUT_TABLE).count(), f"(strategy={write_strategy})")


## Point Power BI at the curated table

In the template, the `Chat + Agent Interactions (Audit Logs)` partition M becomes a thin,
**foldable** passthrough (no JSON parse, no expand, no joins):

```m
let
    Source   = FabricTable("copilot_interactions_curated"),
    Filtered = Table.SelectRows(Source, each [CreationDate] >= RangeStart and [CreationDate] < RangeEnd)
in
    Filtered
```

Because the only step is a range filter on `CreationDate`, it folds to the Lakehouse SQL
endpoint and Incremental Refresh only touches new day-partitions. On Fabric / Premium
capacity you can instead flip the model to **Direct Lake** and skip refresh entirely.